# 05 · 체크포인트에서 MIDI 생성
원본: 배포ver1.2.ipynb


DCGAN 모델 및 학습 루프는 TensorFlow Authors의 DCGAN 튜토리얼을 바탕으로
음악 이미지에 맞게 수정한 프로젝트 코드입니다. Copyright 2019 The TensorFlow Authors.
해당 기반 코드에는 Apache License 2.0이 적용됩니다. 저장소의 THIRD_PARTY_NOTICES.md와
LICENSES/Apache-2.0.txt를 참고하세요. 공개용 정리 과정에서 경로·셀 순서·설명을 수정했습니다.


배포본에 지정된 ckpt-80을 이용하는 추론 흐름입니다. 가중치는 별도로 준비해야 합니다. 기본 과정은 MIDI 저장까지이며, 오디오 미리듣기는 선택 셀입니다. 원본의 이진화·음역 복원·노트 병합 알고리즘을 보존했습니다.

In [ ]:
%pip install midiutil
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import midiutil
from pathlib import Path
from datetime import datetime
from IPython import display
from google.colab import files

In [ ]:
#@title
def make_generator_model():
    model = tf.keras.Sequential()
    model.add(layers.Dense(8*8*256, use_bias=False, input_shape=(100,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((8, 8, 256)))
    assert model.output_shape == (None, 8, 8, 256) # 주목: 배치사이즈로 None이 주어집니다.

    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    assert model.output_shape == (None, 8, 8, 128)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    # assert model.output_shape == (None, 16, 16, 64)
    # model.add(layers.BatchNormalization())
    # model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 16, 16, 64)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(32, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 32, 32, 32)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())


    model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    assert model.output_shape == (None, 64, 64, 1)

    return model


def make_discriminator_model():
    model = tf.keras.Sequential()
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same',
                                     input_shape=[64, 64, 1]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

generator = make_generator_model()
discriminator = make_discriminator_model()

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

In [ ]:
BASE = Path('/content/project')
ckpt = 80
ckpt_path = str(BASE / 'checkpoints' / f'ckpt-{ckpt}')
OUTPUT_DIR = BASE / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not Path(ckpt_path + '.index').exists() or not list(Path(ckpt_path).parent.glob(Path(ckpt_path).name + '.data-*')):
    raise FileNotFoundError('같은 체크포인트의 .index와 .data 파일을 함께 준비하세요.')
status = checkpoint.restore(ckpt_path)
status.assert_existing_objects_matched()
n = 0
midi_bpm = 100

## 새 음악 소재 생성
아래 셀부터 다시 실행하면 새로운 4마디 소재를 만듭니다.

In [ ]:
n+=1 #시행횟수(만들어지는 파일의 이름이 중복됨을 방지)
time = datetime.today().strftime("%Y%m%d%H%M%S") #시간적으려고..(생각해보니 위에거 안해도 됐음..)
noise = tf.random.normal([1, 100]) #정규분포 노이즈
# noise = tf.random.uniform([1,100], minval=-1, maxval=1, dtype=tf.int32)
# noise = tf.cast(noise, tf.float32)
generated_image = generator(noise, training=False)

plt.imshow(generated_image[0, :, :, 0], cmap='gray', origin='lower')

In [ ]:
np.set_printoptions(threshold=np.inf, linewidth=np.inf)
target = generated_image.numpy()
target = target.reshape(64,64)
target = (target+1)/2
target = np.around(target)
plt.imshow(target[:,:], cmap='gray', origin='lower')

In [ ]:
#@title
reshaped_data2 = target

for i in range(36):
  reshaped_data2 = np.insert(reshaped_data2,0,0, axis=0)
for i in range(12):
  reshaped_data2 = np.insert(reshaped_data2,60,0, axis=0)
for i in range(16):
  reshaped_data2 = np.insert(reshaped_data2,112,0, axis=0)


sequence = []
stack = 0
for pit, note in enumerate(reshaped_data2):
  for i,j in enumerate(note):
    if j != 0 and i != 63:
      if note[i+1] != 0:
        stack = stack + 1
        if i+1 == 63: #다음 노트가 맨 뒤 노트일때
          sequence.append({
          'pitch' : pit,
          'start_time' : (i+1-stack)/4,
          'length' : (stack+1)*0.25,
          'velocity' : 80
          })
          stack=0
      else:
        sequence.append({
          'pitch' : pit,
          'start_time' : (i-stack)/4,
          'length' : (stack+1)*0.25,
          'velocity' : 80
          })
        stack=0
    elif j !=0 and i == 63 and note[i-1] == 0:
      sequence.append({
          'pitch' : pit,
          'start_time' : i/4,
          'length' : 0.25,
          'velocity' : 80
        })
      stack=0


def write_midi(seq, bpm, path):
    mf = midiutil.MIDIFile(1, file_format=1)
    track = 0
    channel = 0
    mf.addTempo(track, 0, bpm)
    for i, n in enumerate(seq):
        mf.addNote(
            track,
            channel,
            int(n['pitch']),
            n['start_time'],
            n['length'],
            int(n['velocity'])
        )
    with open(path, 'wb') as outf:
        mf.writeFile(outf)

print(f'{reshaped_data2.shape}로 바꿨음!')
print('변신준비 완료!')

write_midi(sequence, midi_bpm, str(OUTPUT_DIR / f'{time}_ckpt-{ckpt}_{n}.mid'))

print('미디로 씀!')

In [ ]:
midi_path = OUTPUT_DIR / f'{time}_ckpt-{ckpt}_{n}.mid'
files.download(str(midi_path))

## 선택 · 소리로 들어보기
다음 두 셀은 오디오 미리듣기를 원할 때만 실행합니다. FluidSynth와 General MIDI 사운드폰트가 필요하며, 이 음색은 발매곡의 편곡·보컬을 재현하지 않습니다.

In [ ]:
%pip install midi2audio
!apt-get -qq update
!apt-get -qq install -y fluidsynth fluid-soundfont-gm

In [ ]:
from midi2audio import FluidSynth
soundfont = Path('/usr/share/sounds/sf2/FluidR3_GM.sf2')
if not soundfont.exists():
    raise FileNotFoundError('설치한 사운드폰트의 경로를 지정하세요.')
wav_path = midi_path.with_suffix('.wav')
FluidSynth(sound_font=str(soundfont)).midi_to_audio(str(midi_path), str(wav_path))
display.Audio(str(wav_path))